# HIVES и DHF в Google Colab

Развёртывание проекта и запуск тестов с разными комбинациями **экспертов и критериев** (5e6c, 5e8c, 7e6c, 7e8c), как в разделе 3.7 README.

Результаты: `Result/5e6c/`, `Result/5e8c/`, `Result/7e6c/`, `Result/7e8c/` с файлами `*ga.json`, `*hho.json`.

## 1. Установка: клонирование и зависимости

In [ ]:
!git clone https://github.com/Karperash/Metod-HIVES-and-DHF.git
%cd Metod-HIVES-and-DHF
!pip install -q -r requirements.txt

## 2. (По желанию) Быстрые проверки: только HIVES и только DHF

In [ ]:
# Только HIVES:
!python main.py hives examples/hives/input.json

# Только DHF (генерирует my_dhfs_data.json, comparison_results.json):
# !python legacy/main4.py

## 3. Подготовка к тестам 5e6c, 5e8c, 7e6c, 7e8c

Создаём каталоги и добавляем путь к проекту для импорта.

In [ ]:
import os
import json
import sys
import subprocess
from pathlib import Path

# Каталоги из раздела 3.7 README
for d in ["Result/5e6c", "Result/5e8c", "Result/7e6c", "Result/7e8c", "outputs"]:
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, os.getcwd())
import main

print("Каталоги созданы, main загружен.")

## 4. Генерация combined JSON и запуск pipeline

Функция для одной комбинации и одного прогона:
- генерирует combined (hives+dhf) через `main._generate_hives_payload` и `main._generate_dhf_payload`;
- сохраняет `hives` во временный файл, запускает `main.py hives ... -o outputs/prev_<key>_<run>.json` для lambdas предшественника;
- добавляет `rotation` и `combined_parameters` (step2, output_path_ga, output_path_hho);
- сохраняет combined и запускает `main.py pipeline ...`.

In [ ]:
def config_to_experts_criteria(key):
    """ 5e6c -> (5,6), 5e8c -> (5,8), 7e6c -> (7,6), 7e8c -> (7,8) """
    a, b = key.lower().split("e")
    return int(a), int(b.replace("c", ""))

def build_and_run_pipeline(key, run, base_seed=42):
    E, C = config_to_experts_criteria(key)
    seed = base_seed + run
    criteria = [f"Criterion_{i+1}" for i in range(C)]
    expert_ids = [f"DM{i+1}" for i in range(E)]
    alts = [f"A{i+1}" for i in range(3)]

    hives = main._generate_hives_payload(criteria, expert_ids, alts, seed=seed)
    dhf = main._generate_dhf_payload(criteria, expert_ids, seed=seed)
    combined = {"hives": hives, "dhf": dhf, "combined_parameters": {"dhf_method": "HHO"}}

    # HIVES по hives для lambdas предшественника (predecessor_id=DM1)
    hives_path = f"outputs/hives_only_{key}_run{run}.json"
    prev_path = f"outputs/prev_{key}_run{run}.json"
    Path(hives_path).write_text(json.dumps(hives, ensure_ascii=False, indent=2), encoding="utf-8")
    subprocess.run(["python", "main.py", "hives", hives_path, "-o", prev_path], check=True, capture_output=True)

    # rotation и combined_parameters для pipeline (new_id — один из экспертов, остающихся после удаления DM1)
    combined["rotation"] = {
        "predecessor_id": "DM1",
        "new_id": "DM2",
        "alpha": 0.5,
        "predecessor_old_lambdas_path": prev_path,
    }
    combined["combined_parameters"].update({
        "step2_output_path": f"outputs/step2_{key}_run{run}.json",
        "output_path_ga": f"Result/{key}/{key}{run}ga.json",
        "output_path_hho": f"Result/{key}/{key}{run}hho.json",
    })

    combined_path = f"outputs/combined_{key}_run{run}.json"
    Path(combined_path).write_text(json.dumps(combined, ensure_ascii=False, indent=2), encoding="utf-8")
    r = subprocess.run(["python", "main.py", "pipeline", combined_path], capture_output=True, text=True)
    print(r.stdout or "")
    if r.returncode != 0:
        print(r.stderr or "")
        raise subprocess.CalledProcessError(r.returncode, r.cmd, r.stdout, r.stderr)
    print(f"Готово: {key} run{run} -> Result/{key}/{key}{run}ga.json, {key}{run}hho.json")

## 5. Запуск тестов

- **Демо:** `CONFIGS = ["5e6c"]`, `N_RUNS = 1`.
- **Расширенный:** несколько конфигов и/или `N_RUNS` 2–10 (каждый pipeline запускает GA и HHO — это занимает время).

In [ ]:
CONFIGS = ["5e6c"]       # или ["5e6c", "5e8c", "7e6c", "7e8c"]
N_RUNS = 1               # 1–10 прогонов на конфиг
BASE_SEED = 42

for key in CONFIGS:
    for run in range(1, N_RUNS + 1):
        build_and_run_pipeline(key, run, base_seed=BASE_SEED)

print("Все прогоны завершены. Результаты в Result/<конфиг>/*.json")

## 6. (По желанию) Скачать папку Result

In [ ]:
try:
    from google.colab import files
    import shutil
    shutil.make_archive("Result", "zip", "Result")
    files.download("Result.zip")
    print("Скачивание Result.zip запущено.")
except Exception as e:
    print("Скачивание недоступно (не Colab или ошибка):", e)